**[🏠 Course Home](00_START_HERE.ipynb)** | **📖 [Conceptual](../conceptual/21_framework_reliability_and_tolerance_contracts.ipynb)** | **🤖 [AI-Native Track](README.md)** | **🐍 [Python](../python/21_framework_reliability_and_tolerance_contracts.ipynb)** | **🔵 [R](../r/21_framework_reliability_and_tolerance_contracts.ipynb)** | **⬅️ [Prev: Ch 20: CI/CD Test Promotion Gatekeeping](20_test_promotion_heuristics_vs_bayes.ipynb)** | **➡️ [Next: Ch 22: Active Learning & Culprit Search](22_information_theory_and_culprit_search.ipynb)**

*🧭 Part V: Production Reliability, CI/CD & Active Learning — Chapter 21 of 22*

---

# Chapter 21: Framework Reliability & Tolerable Flakiness — The Two-Tier Tolerance Contract
### *Serial Compounding, The False Discovery Rate, and Qualifying Test Harnesses under Real Uncertainty*

> **🧭 Course Context & Why This Module Matters in the AI Era:**  
> Infrastructure architects and platform teams increasingly use AI assistants
> to draft Architecture Decision Records (ADRs) and define Service Level
> Objectives (SLOs) for foundational test harnesses. When asked to specify
> "tolerable flakiness" for a test framework, LLMs naively accept 99.9%
> per-test reliability as "near perfect." This module reveals the catastrophic
> multiplicative collapse of serial test suites, exposes how false alarms
> destroy developer trust via the False Discovery Rate, and teaches agents how
> to enforce mathematically sound Two-Tier Posterior Predictive Contracts.



## Part 1: The Naive Prompt Trap & Forensic Post-Mortem

### The Prompt an Engineer Types
A platform architect prompts an AI assistant while writing an ADR for a new
shared test runner:

```text
"We are launching a new end-to-end testing framework for our microservices.
If our framework guarantees that each individual test invocation has a 99.9%
success rate (0.1% flake rate), is our 500-test CI suite reliable enough for
production deployment?"
```

### The Plausible-Sounding AI Hallucination
The AI assistant responds with glowing reassurance:

> *"Yes, a 99.9% reliability rate is fantastic! Having three nines of
> reliability per test means that failures will be exceedingly rare. Your
> 500-test suite will be exceptionally stable, and developer workflows will
> not be disrupted. You can confidently sign off on this ADR!"*

```text
              THE "THREE NINES" MULTIPLICATIVE TRAP (AI HALLUCINATION)
              
   Per-Test Reliability: 99.9% ──► AI: "Three nines is fantastic!"
                                               │
                                               ▼
   Suite Survival across K = 500 Tests: (0.999)^500 = 60.6% !
   >>> 39.4% OF ALL PULL REQUESTS TURN FALSE-RED! PIPELINE PARALYZED!
```

### Forensic Breakdown: Why the AI Failed
The AI's advice causes immediate operational paralysis because it ignores three
fundamental mathematical laws of testing infrastructure:

1. **Multiplicative Serial Compounding**:
   A test suite is a **serial dependency chain**. A pull request stays green if
   and only if *every single test invocation executes without a framework
   crash*. Across $K = 500$ tests, overall suite survival is:
   $$R_{\text{suite}} = (1 - \theta)^K = (1 - 0.001)^{500} = (0.999)^{500} \approx \mathbf{60.6\%}$$
   **$39.4\%$ of all PR builds will fail purely due to test harness noise!**
   If the suite grows to $K = 1{,}000$ tests, suite survival drops to:
   $$R_{\text{suite}} = (0.999)^{1{,}000} \approx \mathbf{36.8\%}$$
   Nearly two out of three builds will turn red when application code is 100%
   correct!

2. **The Bayesian False Discovery Rate (FDR) Explosion**:
   In engineering organizations, genuine application bugs occur on only a small
   fraction of PRs (e.g., $P(\text{Bug}) = 2.0\%$). If harness infrastructure
   causes false failures on $1.0\%$ of PRs ($P(\text{Flake}) = 0.01$):
   $$\text{False Discovery Rate (FDR)} = \frac{P(\text{Flake})}{P(\text{Bug}) + P(\text{Flake})} = \frac{0.01}{0.02 + 0.01} = \mathbf{33.3\%}$$
   **One out of every three red builds is a false alarm!**
   When FDR exceeds $10\%$, developers suffer alarm fatigue: they reflexively
   re-run builds or bypass red tests, destroying the entire utility of CI.

3. **Per-Test Budget Inversion**:
   To guarantee that $99\%$ of PR builds pass cleanly ($R_{\text{suite}} \ge
   0.99$) across $K = 1{,}000$ tests, the per-invocation error budget must be:
   $$\theta_{\text{test}} \le 1 - R_{\text{suite}}^{1/K} \approx -\frac{\ln(R_{\text{suite}})}{K} = -\frac{\ln(0.99)}{1{,}000} \approx \mathbf{1.0 \times 10^{-5}} \quad (0.001\%)$$
   The true required reliability is not "three nines" ($99.9\%$), but **five
   nines** ($99.999\%$)!



## Part 2: Prompting as Probabilistic Programming — Declarative Specifications

To eliminate AI statistical hallucinations, we must specify the generative
physics, prior distributions, and decision loss rather than asking vague
questions. Depending on your workflow, you can specify models using three
progressively simpler patterns:

---

### Level 1: The Everyday 3-Sentence Micro-Spec *(Fastest for Chat)*
Copy and paste these 3 plain sentences into any LLM:

```markdown
1. System: Large CI test suite of K = 1,000 independent tests.
2. Trap: Demonstrate why 99.9% per-test reliability causes 39.4% false suite breakage
   (R_suite = 0.999^1000 = 60.6%).
3. Contract: Derive per-test budget theta <= 1.0e-5 to achieve a 99% suite survival
   SLA and cap False Discovery Rate below 5%.
```

---

### Level 2: The Model Formula Shorthand *(Concise Probabilistic DSL)*
For quick prompt engineering with statistical models:

```text
Compounding: R_suite = (1 - theta)^K,  theta_target = 1 - R_suite^{1/K}
Target:      theta <= 1.0e-5 for K = 1,000 tests and R_suite >= 99%
FDR:         FDR = (1 - R_suite) * P(Clean) / Total_Failures <= 5%
```

---

### Level 3: The Socratic Interviewer Meta-Prompt *(Zero Effort)*
If you do not want to write the spec yourself, force the AI to interview you:

> *"Act as a Test Infrastructure Architect. We have 1,000 tests each passing 99.9%
of the time, but our build is red 40% of the time. Ask me 3 questions about
multiplicative compounding, per-test error budgets, and FDR before troubleshooting."*

---

<details>
<summary><b>📋 View Full Production YAML Contract (For Automated CI/CD Pipelines & Autonomous Agents)</b></summary>

```yaml
specification_contract:
  target: "Two-Tier Testing Framework Reliability Contract"
  module: "Chapter 21 Infrastructure Quality Sizing"
  
  suite_context:
    suite_size_k: 1000            # Number of test invocations per PR build
    true_bug_rate: 0.020          # 2.0% of PRs contain real application regressions
    max_tolerable_fdr: 0.050      # Maximum 5.0% False Discovery Rate
    
  tier_1_suite_predictive_guarantee:
    description: "Developer Experience SLA (The Primary North Star)"
    metric: "P(Y_framework_flakes == 0 in K=1,000 runs | Telemetry)"
    target_threshold: 0.990       # 99.0% of PR builds must have 0 framework crashes
    operational_meaning: "Caps harness-induced red builds to < 1 in 100 PRs"
    
  tier_2_invocation_tolerance_bound:
    description: "Harness Engineering Acceptance Criterion (Unit Test Derivative)"
    derived_formula: "theta_max = -ln(R_target) / K = -ln(0.990) / 1000"
    target_failure_rate: 1.005e-5 # <= 0.001005% per invocation
    posterior_confidence_gamma: 0.950
    formal_condition: "P(theta <= 1.0e-5 | Benchmark Telemetry) >= 0.950"
    
  qualification_harness:
    method: "Hydraulic Shake Table (Synthetic Stress Injection)"
    stress_parameters:
      cpu_throttling_cores: 1.0
      socket_jitter_ms: 150
      concurrency_multiplier: 4.0
    stress_amplification_factor: 500.0  # Amplifies 1e-5 latent races to ~0.5%
    benchmark_runs: 200                 # Qualifies harness in <200 runs vs 100,000 idle
    
  continuous_monitoring:
    filter: "Exponential memory decay (gamma = 0.98)"
    alarm_condition: "P(theta > 1.0e-5 | Streaming Telemetry) >= 0.10"
```

</details>


## Part 3: The Calibrated Agent Response & Telemetry

When an AI agent receives the Level 1 Micro-Spec or Level 4 YAML contract from
Part 2, it rejects point-estimate "three nines" slogans and formulates a
rigorous Two-Tier Tolerance Contract. Here is the verified mathematical
derivation and execution telemetry:

### Mathematical Derivation of the Two-Tier Tolerance Contract
1. **Serial Test Compounding**:
   In a serial suite of $K = 1{,}000$ independent tests where each test has
   invocation failure probability $\theta$:
   $$R_{\text{suite}}(K, \theta) = \prod_{i=1}^K (1 - \theta) = (1 - \theta)^K$$
   For small $\theta$, this approximates as:
   $$R_{\text{suite}}(K, \theta) \approx \exp(-K \theta)$$
   If $\theta = 10^{-3}$ ($99.9\%$ per test), suite survival collapses:
   $$R_{\text{suite}}(1000, 10^{-3}) = (1 - 0.001)^{1000} \approx 36.79\%$$
   The suite fails $63.21\%$ of the time from harness instability alone!
2. **Tier 2 Error Budget Inversion (The Five Nines Requirement)**:
   To guarantee Tier 1 SLA ($R_{\text{suite}} \ge 0.990$, or $99.0\%$ suite
   survival), the per-test invocation error budget must be:
   $$\theta \le 1 - R_{\text{suite}}^{1/K} \approx
   -\frac{\ln(R_{\text{suite}})}{K}$$
   $$\theta \le -\frac{\ln(0.990)}{1{,}000} \approx \frac{0.01005}{1{,}000}
   \approx 1.005 \times 10^{-5} \quad (99.999\%)$$
3. **False Discovery Rate (FDR) Bound**:
   Given baseline code bug rate $p_{\text{bug}} = 0.020$, the framework flake
   probability must satisfy:
   $$\text{FDR} = \frac{1 - R_{\text{suite}}}{p_{\text{bug}} + (1 -
   R_{\text{suite}})} \le 0.050$$
   $$\implies 1 - R_{\text{suite}} \le \frac{0.050 \times 0.020}{0.950} \approx
   0.001053$$
   The framework may trigger at most one false red build per 1,000 PRs.
4. **Accelerated Qualification via Hydraulic Shake-Table**:
   We amplify timing race conditions by $A = 500\times$. Observing $N = 200$
   zero-failure stress runs bounds idle failure rate via the Rule of Three:
   $$\theta_{\text{stress}} \le \frac{3}{200} = 0.015 \implies
   \theta_{\text{idle}} \le \frac{0.015}{500} = 3.0 \times 10^{-5}$$

### Worked Agent Execution Transcript & Comparative Telemetry

Below is the verified execution telemetry when the agent evaluates the testing
framework ADR and runs qualification benchmarks:

```text
================================================================================
       FRAMEWORK RELIABILITY AUDITOR: TWO-TIER CONTRACT EVALUATION
================================================================================
[INPUT ADR SPECIFICATION]
  Claimed Harness Reliability: 99.9% per invocation (theta = 1.0e-3)
  Target Test Suite Size:      K = 1,000 tests per PR build
  Baseline PR Bug Rate:        P(Bug) = 2.0%

[MATHEMATICAL FORENSIC AUDIT]
  Multiplicative Suite Survival: R_suite = (1 - 0.001)^1000 = 36.79%
  Harness-Induced Red Builds:    1 - R_suite = 63.21%
  False Discovery Rate (FDR):    0.6321 / (0.020 + 0.6321) = 96.93% !
  >>> CRITICAL REJECTION: 97% of red builds are infrastructure false alarms!

[RECALIBRATED TWO-TIER SPECIFICATION CONTRACT]
  Tier 1 Goal (Suite SLA):       P(Zero Framework Crashes in K=1000) >= 99.0%
  Tier 2 Bound (Per-Test): theta <= -ln(0.99)/1000 = 1.005e-5 (Five Nines!)
  Maximum Allowable FDR:         FDR = 0.010 / (0.020 + 0.010) = 4.76% <= 5.0%

[ACCELERATED QUALIFICATION BENCHMARK]
  Running Hydraulic Shake Table (500x timing stress injection)...
  Simulating 200 runs with 1 core CPU throttle + 150ms socket jitter:
  
  Trial 001..050: PASS  [No socket hangs detected]
  Trial 051..100: PASS  [GC pauses handled cleanly]
  Trial 101..150: PASS  [Connection pooling stable]
  Trial 151..200: PASS  [Teardown hooks verified]
  
  Benchmark Result: 0 failures in 200 runs.
  Rule of Three Stress Bound: theta_stress <= 3 / 200 = 0.015 (1.5%)
  De-amplified Idle Bound:    theta_idle <= 0.015 / 500 = 3.0e-5 (Qualified!)

  ASCII RELIABILITY CURVE (Suite Survival vs. Suite Size K):
    K = 100   | ************************************************** 99.9%
    K = 500   | ************************************************   99.5%
    K = 1,000 | ***********************************************     99.0% [SLA]
    K = 2,500 | ********************************************       97.5%
    K = 5,000 | ****************************************           95.1%

[FINAL VERDICT]: Framework qualified. ADR approved under Two-Tier Contract.
================================================================================
```

### Risk & Decision Evaluation
Under the Two-Tier contract, the framework bounds the False Discovery Rate to
$\text{FDR} \le 4.76\% \le 5.0\%$ and guarantees suite survival
$R_{\text{suite}} \ge 99.0\%$. The ADR is officially approved.

## Part 4: The 4 Mandatory Verification Gates (How to Audit the Agent)

The AI agent returned a clean, calibrated response: it derived the five-nines
Tier 2 bound, calculated the FDR explosion, and qualified the runner via
accelerated stress testing.

However, as an **Epistemic Architect**, you must never accept a testing
infrastructure recommendation without auditing the agent against **four
mandatory verification gates**:

```
              THE 4 MANDATORY AGENTIC VERIFICATION GATES
┌────────────────────────────────────────────────────────────────────┐
│ Gate 1: Prior Predictive Check                                     │
│ Sane physical domain bounds before seeing real data                │
└─────────────────────────────────┬──────────────────────────────────┘
                                  │ [Domain Sanity Passed]
                                  ▼
┌────────────────────────────────────────────────────────────────────┐
│ Gate 2: Simulation-Based Calibration (SBC)                         │
│ Algorithmic proof: recovers ground truth across 5,000 simulations  │
└─────────────────────────────────┬──────────────────────────────────┘
                                  │ [Algorithm Soundness Passed]
                                  ▼
┌────────────────────────────────────────────────────────────────────┐
│ Gate 3: Mathematical Invariant Conservation                        │
│ Shannon entropy non-negativity / Mutual info / Promotion rule      │
└─────────────────────────────────┬──────────────────────────────────┘
                                  │ [Information Theoretic Invariants]
                                  ▼
┌────────────────────────────────────────────────────────────────────┐
│ Gate 4: Posterior Predictive Retrodiction & Loss                   │
│ Sane predictive fit to telemetry & expected loss decision audit    │
└─────────────────────────────────┬──────────────────────────────────┘
                                  │
                                  ▼ [SAFE FOR PRODUCTION DEPLOYMENT]
```

---

### Gate 1: Prior Predictive Check (Serial Compounding & Suite Survival)

* **The Production Risk (Why This Gate Exists)**:
  The single most devastating cognitive trap in software testing is the **local
  metric fallacy**: assuming that an individual test invocation reliability of
  $99.9\%$ ("three nines") is sufficient for production. When an unguided AI
  agent approves an ADR based on per-invocation reliability, it blinds the team
  to serial compounding. In a suite of $K = 1{,}000$ tests, a $99.9\%$ runner
  causes a $63.2\%$ suite failure rate. Developers face continuous red builds,
  development halts, and the test suite is abandoned. Gate 1 forces generative
  verification of compounding curves across suite sizes before touching code.
* **First-Principles Mathematical Mechanism**:
  Let $X_i \sim \text{Bernoulli}(\theta)$ represent harness failure for test
  $i \in \{1, \dots, K\}$. Under independence, suite survival is:
  $$R_{\text{suite}}(K, \theta) = \prod_{i=1}^K P(X_i = 0) = (1 - \theta)^K$$
  The first-order logarithmic expansion gives:
  $$\ln R_{\text{suite}} = K \ln(1 - \theta) \approx -K \theta \implies
  R_{\text{suite}} \approx \exp(-K \theta)$$
  To preserve a target suite survival $R_{\text{suite}}$, the maximum allowable
  per-test failure rate is:
  $$\theta_{\text{max}} = 1 - R_{\text{suite}}^{1/K} \approx
  -\frac{\ln(R_{\text{suite}})}{K}$$
  We evaluate suite survival across expanding test suites:
  * For $K = 100$: $R = (1 - 0.001)^{100} \approx 90.48\%$
  * For $K = 500$: $R = (1 - 0.001)^{500} \approx 60.64\%$
  * For $K = 1{,}000$: $R = (1 - 0.001)^{1000} \approx 36.79\%$
  * For $K = 2{,}500$: $R = (1 - 0.001)^{2500} \approx 8.20\%$
  The agent must prove that achieving $99.0\%$ suite survival at $K = 1{,}000$
  strictly mandates $\theta \le 1.005 \times 10^{-5}$ ("five nines").
* **The Automated Assertion Contract (`[PASS]` / `[FAIL]` Criteria)**:
  * Compounding Identity:
    $|R_{\text{suite}}(1000, 10^{-3}) - 0.367879| < 10^{-6}$ (`[PASS]`).
  * Five-Nines Invariant:
    For $R \ge 0.990, K = 1{,}000$, $\theta_{\text{max}} \le 1.005 \times
    10^{-5}$
    (`[PASS]`).
  * Strictly Monotonic Suite Decay:
    $\frac{\partial R}{\partial K} = (1-\theta)^K \ln(1-\theta) < 0$
    for all $\theta > 0$ (`[PASS]`).
* **The Audit Prompt**:
  ```text
  AUDIT GATE 1 (Serial Compounding & Suite Survival):
  Given test runner with claimed per-test reliability theta = 0.001 (99.9%):
  1. Compute exact suite survival R_suite = (1 - theta)^K for K in [100, 500,
     1000, 2500].
  2. Prove that R_suite collapses to ~36.8% at K = 1000.
  3. Invert the formula to solve for the maximum allowable theta to guarantee
     R_suite >= 0.990 at K = 1000: theta <= -ln(0.99) / 1000 ≈ 1.005e-5.
  Assert exact algebraic equality. Report [PASS] or [FAIL].
  ```

---

### Gate 2: Simulation-Based Calibration (SBC) / False Discovery Recovery

* **The Production Risk (Why This Gate Exists)**:
  The secondary catastrophe of framework flakiness is **developer alarm
  fatigue** governed by the **False Discovery Rate (FDR)**. When a PR fails CI,
  engineers must know whether the failure is a genuine code bug or a test
  harness flake. If the framework causes $63.2\%$ red builds while genuine code
  bugs occur on only $2.0\%$ of PRs, $96.9\%$ of all red builds are false
  alarms! Developers rapidly learn to ignore CI, click "retry", or merge without
  testing. Gate 2 simulates thousands of PR pipelines to prove that the Two-Tier
  contract controls FDR below $5\%$.
* **First-Principles Mathematical Mechanism**:
  We simulate $M = 5{,}000$ synthetic PR batches where true code bugs occur
  with prior probability $B \sim \text{Bernoulli}(p_{\text{bug}} = 0.020)$.
  The test suite executes $K = 1{,}000$ tests. Harness crashes occur with
  probability $F = 1 - (1 - \theta)^K$.
  Total red build probability is:
  $$P(\text{Red}) = P(B=1 \cup F=1) = p_{\text{bug}} + P(F) - p_{\text{bug}}
  P(F)$$
  False Discovery Rate is the posterior probability of no code bug given a red
  build:
  $$\text{FDR} = P(B=0 \mid \text{Red}) = \frac{(1 - p_{\text{bug}})
  P(F)}{P(\text{Red})}$$
  Under naive "three nines" ($\theta = 10^{-3}$), $P(F) = 0.6321$:
  $$\text{FDR}_{\text{naive}} = \frac{0.980 \times 0.6321}{0.020 + 0.6321 -
  0.020 \times 0.6321} = \frac{0.6195}{0.6395} \approx 96.87\%$$
  Under the Two-Tier contract ($P(F) \le 0.00105$):
  $$\text{FDR}_{\text{contract}} = \frac{0.980 \times 0.00105}{0.020 + 0.00105 -
  0.00002} \approx \frac{0.001029}{0.02103} \approx 4.89\% \le 5.0\%$$
  Across 5,000 simulations, the agent proves that the Two-Tier contract bounds
  empirical FDR $\le 5.0\%$.
* **The Automated Assertion Contract (`[PASS]` / `[FAIL]` Criteria)**:
  * Empirical FDR Invariant:
    $\text{FDR} \le 0.050$ across 5,000 simulation batches (`[PASS]`; naive
    runner $\text{FDR} \approx 0.969$ triggers immediate `[FAIL]`).
  * Suite Survival Invariant:
    Empirical suite survival $\ge 0.990$ across all clean PRs (`[PASS]`).
  * Signal-to-Noise Ratio:
    $\frac{P(\text{True Bug})}{P(\text{Harness Flake})} \ge 19.0$ (`[PASS]`).
* **The Audit Prompt**:
  ```text
  AUDIT GATE 2 (Simulation-Based Calibration & FDR Control):
  Simulate 5,000 PR builds with p_bug = 0.020 and K = 1,000 tests:
  1. Compare the naive runner (theta = 0.001) against the Two-Tier contract
     (theta <= 1.005e-5).
  2. Compute empirical False Discovery Rate: FDR = False_Red / Total_Red.
  3. Assert that the Two-Tier contract holds FDR <= 5.0% while the naive runner
     produces FDR > 95%.
  Report [PASS] or [FAIL].
  ```

---

### Gate 3: Mathematical Invariants (Stress De-amplification)

* **The Production Risk (Why This Gate Exists)**:
  Verifying a five-nines failure rate ($\theta \le 10^{-5}$) under standard
  idle conditions requires executing at least $300{,}000$ test runs to observe
  even 3 failures—costing weeks of cluster time and tens of thousands of
  dollars. Teams inevitably skip empirical verification and guess. To qualify
  the harness affordably, we use **synthetic stress injection** (CPU throttling,
  socket jitter) to amplify race conditions by factor $A = 500\times$. Gate 3
  verifies the mathematical scaling invariants and de-amplification bounds,
  ensuring no physical information leaks occur.
* **First-Principles Mathematical Mechanism**:
  1. **Linear Stress Scaling Invariant**:
     Let failure rate under synthetic timing stress be $\theta_{\text{stress}}$
     and idle failure rate be $\theta_{\text{idle}}$. The physical stress model
     posits:
     $$\theta_{\text{stress}} = A \cdot \theta_{\text{idle}} \implies
     \theta_{\text{idle}} = \frac{\theta_{\text{stress}}}{A}$$
     where $A = 500$ is the calibrated acceleration factor.
  2. **Rule of Three Posterior Upper Bound**:
     For $N = 200$ independent trials with zero observed failures ($k = 0$),
     the $95\%$ posterior credible upper bound under a flat prior
     $\text{Beta}(1, 1)$ or Poisson approximation is:
     $$(1 - \theta_{\text{stress}})^N = 0.05 \implies \theta_{\text{stress}} \le
     1 - (0.05)^{1/N} \approx -\frac{\ln(0.05)}{N} = \frac{2.9957}{N} \approx
     \frac{3}{N}$$
     For $N = 200$:
     $$\theta_{\text{stress}} \le \frac{3}{200} = 0.0150 \quad (1.50\%)$$
  3. **De-amplified Idle Invariant**:
     $$\theta_{\text{idle}} \le \frac{\theta_{\text{stress}}}{A} =
     \frac{0.0150}{500} = 3.0 \times 10^{-5}$$
  4. **Information Theoretic Data Processing Inequality**:
     $$I(\theta_{\text{idle}}; Y_{\text{stress}}) \le I(\theta_{\text{stress}};
     Y_{\text{stress}})$$
     No spurious information is created by synthetic stress transformation.
* **The Automated Assertion Contract (`[PASS]` / `[FAIL]` Criteria)**:
  * Stress Scaling Invariant:
    $\theta_{\text{idle}} == \theta_{\text{stress}} / A$ strictly (`[PASS]`).
  * Rule of Three Zero-Failure Invariant:
    $\theta_{\text{stress}} \le 3 / N$ for $k=0$ (`[PASS]`).
  * De-amplified Idle Bound:
    $\theta_{\text{idle}} \le 3.0 \times 10^{-5}$ (`[PASS]`).
* **The Audit Prompt**:
  ```text
  AUDIT GATE 3 (Accelerated Stress De-amplification Invariants):
  Given N = 200 stress runs with 0 failures under acceleration A = 500x:
  1. Apply the Rule of Three to calculate 95% upper bound on theta_stress:
     theta_stress <= 3 / 200 = 0.015.
  2. De-amplify by factor A = 500 to bound idle failure rate:
     theta_idle <= 0.015 / 500 = 3.0e-5.
  3. Confirm that 200 accelerated runs provide equivalent diagnostic power to
     100,000 idle runs.
  Assert exact mathematical inequality. Report [PASS] or [FAIL].
  ```

---

### Gate 4: Posterior Predictive Retrodiction & Loss (ADR Approval Gate)

* **The Production Risk (Why This Gate Exists)**:
  Approving a flawed testing framework has immense asymmetric economic impact:
  a broken runner deployed org-wide generates hundreds of false red builds per
  day, costing thousands of developer hours ($C_{\text{flake}} = \$200$ per
  red build, or $>\$50{,}000$ per week). In contrast, delaying framework
  approval for further hardening costs only $C_{\text{rework}} = \$1{,}000$.
  An unconstrained AI agent that treats "three nines" as passing rubber-stamps
  a catastrophic infrastructure deployment. Gate 4 forces Bayes-optimal decision
  execution under asymmetric risk.
* **First-Principles Mathematical Mechanism**:
  1. **Posterior Predictive Retrodiction**:
     We draw replicated PR build sequences $\tilde{k}_{1:M}$ from the posterior
     predictive distribution:
     $$\tilde{k} \sim \text{Binomial}(K=1000, \theta_{\text{idle}})$$
     For $\theta_{\text{idle}} \le 3.0 \times 10^{-5}$, the probability of
     observing zero framework failures across a $1{,}000$-test suite is:
     $$P(\tilde{k} = 0) = (1 - \theta_{\text{idle}})^{1000} \ge (1 - 3.0 \times
     10^{-5})^{1000} \approx 97.04\%$$
  2. **Asymmetric Decision Contract**:
     Action space $\mathcal{A} = \{\text{Approve\_ADR}, \text{Reject\_ADR}\}$.
     Loss function:
     *  $L(\text{Approve}, \text{unstable}) = C_{\text{flake\_disaster}} =
       50{,}000$
     * $L(\text{Approve}, \text{stable}) = 0$
     * $L(\text{Reject}, \text{unstable}) = 0$
     * $L(\text{Reject}, \text{stable}) = C_{\text{rework}} = 1{,}000$
     The framework is approved if and only if both Tier 1 and Tier 2 criteria
     are proven:
     $$P(R_{\text{suite}} \ge 0.990 \mid \mathcal{D}) \ge 0.950 \quad \text{AND}
     \quad \text{FDR} \le 0.050$$
     Because the benchmark demonstrated $\theta_{\text{idle}} \le 3.0 \times
     10^{-5}$,
     the expected loss of approval is zero, and the Bayes-optimal decision is
     **APPROVE_ADR**.
* **The Automated Assertion Contract (`[PASS]` / `[FAIL]` Criteria)**:
  * Retrodictive Zero-Failure Mass:
    $P(\tilde{k} = 0 \mid K=1000) \ge 0.970$ (`[PASS]`).
  * Dual-Tier Gate Enforcement:
    `action == APPROVE_ADR` if and only if Tier 1 and Tier 2 are satisfied
    (`[PASS]`; approving $\theta = 10^{-3}$ triggers immediate `[FAIL]`).
* **The Audit Prompt**:
  ```text
  AUDIT GATE 4 (Bayesian ADR Approval & Predictive Retrodiction):
  Given benchmark evidence theta_idle <= 3.0e-5 and suite size K = 1,000:
  1. Compute posterior predictive probability of zero harness crashes in K=1000:
     P(zero crashes) >= 97.0%.
  2. Verify expected False Discovery Rate FDR <= 5.0%.
  3.  Evaluate asymmetric loss: L(Approve | unstable) = 50,000 vs L(Reject) =
     1,000.
  4. Assert decision == APPROVE_ADR.
  Report [PASS] or [FAIL].
  ```

## Part 5: Naive AI Prompt vs. Calibrated Epistemic Specification

| Evaluation Metric | Naive AI Prompting ("Three Nines") | Calibrated Epistemic Specification (Two-Tier Contract) |
| :--- | :--- | :--- |
| **Reliability Scope** | Local point metric (single test invocation) | End-to-end system metric (entire serial suite of $K$ tests) |
| **Suite Pass Rate ($K = 1{,}000$)** | **$36.8\%$ survival** ($63.2\%$ false red builds) | **$99.0\%$ guaranteed suite survival** |
| **False Discovery Rate (FDR)** | **$96.9\%$** (developers ignore all test failures) | **$\le 4.8\%$** (preserves total developer trust) |
| **Required Test Reliability** | $\theta = 10^{-3}$ ($99.9\%$, "Three Nines") | $\theta \le 1.0 \times 10^{-5}$ ($99.999\%$, "Five Nines") |
| **Self-Scaling Capacity** | Fixed number; fails when test suite expands | Automatically tightens bound ($\theta \le -\ln(R) / K$) as $K$ grows |
| **Qualification Feasibility** | Requires $100{,}000$ idle runs (unaffordable) | Requires $<200$ runs via synthetic stress injection ($500\times$) |



## Part 6: 🎯 Check Your Understanding — 3-Tier "Audit the AI" Exercises

### 🟢 Tier 1: Spot the AI Hallucination
An AI agent reviews an infrastructure RFC:
> *"Our test suite has grown from 200 tests to 2,000 tests. Because our test
> runner maintains a solid 99.8% reliability per test, this 10x growth in test
> count will have zero impact on developer CI success rates."*

**Your Task**: Calculate the suite survival before and after the 10x suite
expansion and explain why the AI's statement is mathematically false.

<details><summary><b>Click here to view the solution...</b></summary>

**Solution**:
1. **Suite Survival at $K = 200$**:
   $$R_{200} = (1 - 0.002)^{200} = (0.998)^{200} \approx \mathbf{67.0\%}$$
2. **Suite Survival at $K = 2{,}000$**:
   $$R_{2{,}000} = (1 - 0.002)^{2{,}000} =
   (0.998)^{2{,}000} \approx \mathbf{1.8\%!}$$
3. **The Disaster**: With 200 tests, a third of builds failed due to harness
   flakiness. With 2,000 tests, **over 98% of all PR builds will turn red**! A
   10x growth in test count without tightening per-test reliability causes total
   pipeline collapse.
</details>

---

### 🟡 Tier 2: The Everyday 3-Sentence Micro-Spec
An infrastructure engineer needs to qualify a new test execution harness for a
suite of $K = 1{,}500$ tests. The platform target is that $\ge 99.0\%$ of
suite runs must be free of harness flakes ($R_{\text{suite}} \ge 0.990$). The
true PR bug rate is $P(\text{Bug}) = 1.5\%$, and team SLA requires False
Discovery Rate $\text{FDR} \le 5.0\%$. To qualify the runner without waiting
for millions of idle runs, the team uses a Hydraulic Shake Table with
acceleration factor $A = 500\times$.

**Your Task**: Write an everyday 3-sentence micro-spec (`Data`, `Prior`,
`Loss/Action`) for an LLM to derive the allowable per-test flake budget
$\theta_{\text{max}}$ and the required number of clean stress trials. In your
solution, show the exact prompt, the mathematical calculation, and the
resulting action decision.

<details><summary><b>Click here to view the solution...</b></summary>

**1. The 3-Sentence Micro-Spec**:
```markdown
1. Data: Suite size K = 1500 tests, baseline PR code bug rate P(Bug) = 0.015,
   and target FDR <= 0.050.
2. Prior: Harness failure follows Bernoulli(theta) per test, accelerated
   by factor A = 500x under hydraulic stress injection.
3. Loss/Action: Derive allowable per-test flake rate theta_max under the FDR
   cap, and apply the Rule of Three to calculate required clean stress runs
   N_stress to emit QUALIFY_RUNNER.
```

**2. Mathematical Calculation**:
* **Permissible Suite Flake Rate from FDR**:
  $$\text{FDR} =
  \frac{P(\text{Flake})}{P(\text{Bug}) + P(\text{Flake})} \le 0.050$$
  $$P(\text{Flake}) \le \frac{0.050 \times 0.015}{1 - 0.050} =
  \frac{0.00075}{0.950} \approx 7.89 \times 10^{-4}$$
  Target suite survival:
  $R_{\text{suite}} = 1 - P(\text{Flake}) \ge 0.99921$.
* **Per-Test Reliability Budget $\theta_{\text{max}}$**:
  $$\theta_{\text{max}} = -\frac{\ln(R_{\text{suite}})}{K} \approx
  \frac{7.89 \times 10^{-4}}{1500} \approx 5.26 \times 10^{-7}$$
* **Stress-Accelerated Benchmark Sizing (Rule of Three)**:
  Under stress acceleration $A = 500\times$, the amplified failure rate is:
  $$\theta_{\text{stress}} = A \times \theta_{\text{max}} =
  500 \times (5.26 \times 10^{-7}) \approx 2.63 \times 10^{-4}$$
  Under the Rule of Three ($95\%$ confidence with zero failures):
  $$N_{\text{stress}} =
  \left\lceil \frac{3}{\theta_{\text{stress}}} \right\rceil =
  \left\lceil \frac{3}{2.63 \times 10^{-4}} \right\rceil \approx
  \mathbf{11{,}400 \text{ runs}}$$
  *(Note: Sizing against baseline $R_{\text{suite}} \ge 0.990$ alone yields
  $\theta = 6.70 \times 10^{-6}$ and $N_{\text{stress}} \approx 900$ runs).*

**3. Action Decision**:
`RUN_STRESS_BENCHMARK(N=11400)`. If zero harness failures occur across the
accelerated stress runs, emit `QUALIFY_RUNNER`. If $\ge 1$ failure occurs, emit
`REJECT_RUNNER` and isolate concurrency race conditions.
</details>

---

### 🔴 Tier 3: Interrogate the Autonomous Agent
An autonomous infrastructure agent evaluates a test harness and approves an
Architecture Decision Record (ADR):
> *"The candidate test runner completed 500 benchmark runs with 0 crashes.
> Under the Rule of Three, failure probability is bounded to theta <= 3/500 =
> 0.006 (99.4% per-test reliability). Since 99.4% exceeds our 99.0% quality
> SLA, I have approved the ADR for our 1,500-test production CI suite."*

**Your Task**: Formulate a 2-line follow-up audit prompt to catch the agent's
suite survival compounding fallacy and FDR explosion across Gate 1 (Serial
Compounding) and Gate 4 (Bayesian Loss & FDR), and explain the exact invariants
that invalidate the approval.

<details><summary><b>Click here to view the solution...</b></summary>

**1. The 2-Line Follow-Up Audit Prompt**:
```text
Project suite survival R = (1 - theta)^1500 for theta = 0.006 and calculate
the resulting suite flake probability P_flake.
Compute False Discovery Rate FDR given P(Bug) = 0.015 and audit ADR approval
against the FDR <= 5% SLA.
```

**2. The Mathematical Invariants That Catch the Failure**:
* **Gate 1 Compounding Invariant**:
  Harness unreliability compounds exponentially across tests. For $K = 1{,}500$:
  $$R_{\text{suite}} = (1 - 0.006)^{1500} \approx
  \exp(-1500 \times 0.006) = \exp(-9.0) \approx \mathbf{0.012\%}$$
  The suite survival is essentially zero: **$99.98\%$ of PR builds will fail**
  due to harness crashes alone!
* **Gate 4 FDR Invariant**:
  With suite flake probability
  $P(\text{Flake}) = 1 - R_{\text{suite}} \approx 0.99988$
  and true bug rate $P(\text{Bug}) = 0.015$:
  $$\text{FDR} =
  \frac{P(\text{Flake})}{P(\text{Bug}) + P(\text{Flake})} =
  \frac{0.99988}{0.015 + 0.99988} \approx \mathbf{98.5\%}$$
  Out of every 100 red builds, 98 are false alarms.
* **The Root Fallacy**:
  The agent committed the **local metric fallacy**: confusing a per-test
  reliability of $99.4\%$ with a whole-suite reliability SLA. Approving the ADR
  based on 500 isolated runs causes catastrophic infrastructure collapse.
</details>

## Key Takeaways & Course Navigation

1. **Serial Compounding Kills Suites**: A 99.9% reliable test runner collapses a
   500-test suite to 60.6% survival and a 1,000-test suite to 36.8% survival.
2. **The False Discovery Rate Destroys Trust**: If harness flakes occur on 1% of
   PRs and real bugs occur on 2% of PRs, 33.3% of red builds are false alarms,
   triggering developer alarm fatigue.
3. **The Two-Tier Contract Solves the Dilemma**: Specify Tier 1 (99% suite
   survival) for developer SLAs, and derive Tier 2 ($\theta \le 10^{-5}$) for
   engineering benchmarks.
4. **Stress Injection Enables Fast Qualification**: A Hydraulic Shake Table
   amplifies latent timing races by $500\times$, qualifying a framework in
$<200$
   runs instead of $100{,}000$ idle runs.

---

## 🧭 Course Navigation

* 🏠 **[Course Home & Master Syllabus](00_START_HERE.ipynb)**
* ⬅️ **Previous**: **[Ch 20: CI/CD Test Promotion Gatekeeping](20_test_promotion_heuristics_vs_bayes.ipynb)** *(Part V: Production Reliability, CI/CD & Active Learning)*
* ➡️ **Next**: **[Ch 22: Active Learning & Culprit Search](22_information_theory_and_culprit_search.ipynb)** *(Part V: Production Reliability, CI/CD & Active Learning)*
* 🔄 **Companion Tracks for Chapter 21**: [📖 Conceptual](../conceptual/21_framework_reliability_and_tolerance_contracts.ipynb) | [🤖 AI-Native Track](README.md) | [🐍 Python](../python/21_framework_reliability_and_tolerance_contracts.ipynb) | [🔵 R](../r/21_framework_reliability_and_tolerance_contracts.ipynb)

